In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [1]:
# Household type by tenure
# To download original dataset go to - 
# https://www.nomisweb.co.uk/query/construct/submit.asp?menuopt=201&subcomp=
household_type_csv = r"N:\Geodatabase\Raw_Data\Census 2021\Household Type and Tenure\RM003 - Accommodation type by type of central heating in household by tenure.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [2]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\UK DATA\2021 Census Data\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [3]:
threshold_value = 5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [6]:
# Create Dataframes
househole_type_all_df = pd.read_csv(household_type_csv, skiprows = 8, nrows = 35672, skip_blank_lines = True)
househole_type_owned_outright_df = pd.read_csv(household_type_csv, skiprows = 8, nrows = 35672, skip_blank_lines = True)
househole_type_owned_mortgage_df = pd.read_csv(household_type_csv, skiprows = 8, nrows = 35672, skip_blank_lines = True)
househole_type_rented_social_df = pd.read_csv(household_type_csv, skiprows = 8, nrows = 35672, skip_blank_lines = True)
househole_type_rented_private_df = pd.read_csv(household_type_csv, skiprows = 8, nrows = 35672, skip_blank_lines = True)
#male_pop_df = pd.read_csv(sex_by_age_csv, skiprows = 35703, nrows = 35672, skip_blank_lines = True)

In [7]:
# Split the first column into two new columns
female_pop_df[['lsoa21cd', 'lsoa21nm']] = female_pop_df.iloc[:, 0].str.split(' : ', expand=True)
male_pop_df[['lsoa21cd', 'lsoa21nm']] = male_pop_df.iloc[:, 0].str.split(' : ', expand=True)

# Remove the column '2021 super output area - lower layer'
female_pop_df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
male_pop_df.drop(['2021 super output area - lower layer'], 1, inplace=True)

# Rename columns for female_pop_df
cols_to_rename1 = female_pop_df.columns.difference(['lsoa21cd', 'lsoa21nm'])
female_pop_df.rename(columns={col: col.lower().replace(' ', '_') + '_female_count' for col in cols_to_rename1}, inplace=True)

# Rename columns for male_pop_df
cols_to_rename2 = male_pop_df.columns.difference(['lsoa21cd', 'lsoa21nm'])
male_pop_df.rename(columns={col: col.lower().replace(' ', '_') + '_male_count' for col in cols_to_rename2}, inplace=True)

# Remove the column '2021 super output area - lower layer'
female_pop_df.drop(['lsoa21nm'], 1,  inplace=True)
male_pop_df.drop(['lsoa21nm'], 1, inplace=True)

# Move 'lsoa21cd' and 'lsoa21nm' to the front of the dataframe
female_pop_df = female_pop_df[['lsoa21cd'] + [col for col in female_pop_df.columns if col not in ['lsoa21cd']]]
male_pop_df = male_pop_df[['lsoa21cd'] + [col for col in male_pop_df.columns if col not in ['lsoa21cd']]]

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\3103369898.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  female_pop_df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\3103369898.py:7: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  male_pop_df.drop(['2021 super output area - lower layer'], 1, inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\3103369898.py:18: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  female_pop_df.drop(['lsoa21nm'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\3103369898.py:19: FutureWarning: In a future version of pandas all arguments of DataFrame.drop e

## 2. Feature Engineering

In [8]:
# Create percentage columns for female_pop_df
total_female_col = 'total_female_count'
for col in female_pop_df.columns[-23:]:
    perc_col = col.replace('_count', '_perc')
    female_pop_df[perc_col] = (female_pop_df[col] / female_pop_df[total_female_col]) * 100

# Create percentage columns for male_pop_df
total_male_col = 'total_male_count'
for col in male_pop_df.columns[-23:]:
    perc_col = col.replace('_count', '_perc')
    male_pop_df[perc_col] = (male_pop_df[col] / male_pop_df[total_male_col]) * 100

In [9]:
# Combine the two dataframes on lsoa21cd and lsoa21nm
combined_df = female_pop_df.merge(male_pop_df, on=['lsoa21cd'])

# Display the first few rows of the combined dataframe
combined_df.head()

,lsoa21cd,total_female_count,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,total_male_count,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,aged_3_to_4_years_male_perc,aged_5_to_7_years_male_perc,aged_8_to_9_years_male_perc,aged_10_to_14_years_male_perc,aged_15_years_male_perc,aged_16_to_17_years_male_perc,aged_18_to_19_years_male_perc,aged_20_to_24_years_male_perc,aged_25_to_29_years_male_perc,aged_30_to_34_years_male_perc,aged_35_to_39_years_male_perc,aged_40_to_44_years_male_perc,aged_45_to_49_years_male_perc,aged_50_to_54_years_male_perc,aged_55_to_59_years_male_perc,aged_60_to_64_years_male_perc,aged_65_years_male_perc,aged_66_to_69_years_male_perc,aged_70_to_74_years_male_perc,aged_75_to_79_years_male_perc,aged_80_to_84_years_male_perc,aged_85_years_and_over_male_perc
0,E01011954,1200,42,41,40,33,82,11,31,22,70,76,96,83,61,80,91,81,70,8,48,49,25,32,28,3.500000,3.416667,3.333333,2.750000,6.833333,0.916667,2.583333,1.833333,5.833333,6.333333,8.000000,6.916667,5.083333,6.666667,7.583333,6.750000,5.833333,0.666667,4.000000,4.083333,2.083333,2.666667,2.333333,1080,40,43,51,42,81,18,26,19,73,77,75,56,69,47,78,80,71,9,31,42,20,18,14,3.703704,3.981481,4.722222,3.888889,7.500000,1.666667,2.407407,1.759259,6.759259,7.129630,6.944444,5.185185,6.388889,4.351852,7.222222,7.407407,6.574074,0.833333,2.870370,3.888889,1.851852,1.666667,1.296296
1,E01011969,681,19,9,12,14,24,15,17,5,30,23,43,35,31,34,55,41,54,14,48,57,38,39,24,2.790015,1.321586,1.762115,2.055800,3.524229,2.202643,2.496329,0.734214,4.405286,3.377386,6.314244,5.139501,4.552129,4.992658,8.076358,6.020558,7.929515,2.055800,7.048458,8.370044,5.580029,5.726872,3.524229,663,17,16,16,12,35,6,8,15,33,20,29,39,25,28,50,50,63,8,37,58,44,26,28,2.564103,2.413273,2.413273,1.809955,5.279035,0.904977,1.206637,2.262443,4.977376,3.016591,4.374057,5.882353,3.770739,4.223228,7.541478,7.541478,9.502262,1.206637,5.580694,8

In [10]:
# Create aggregated count columns by summing female and male counts
count_columns = [col for col in combined_df.columns if col.endswith('_female_count')]
for col in count_columns:
    base_name = col.replace('_female_count', '_count')
    male_col = col.replace('_female_count', '_male_count')
    if male_col in combined_df.columns:
        combined_df[base_name] = combined_df[col] + combined_df[male_col]

# Display the first few rows of the combined dataframe
combined_df.head()

,lsoa21cd,total_female_count,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,total_male_count,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,aged_3_to_4_years_male_perc,aged_5_to_7_years_male_perc,aged_8_to_9_years_male_perc,aged_10_to_14_years_male_perc,aged_15_years_male_perc,aged_16_to_17_years_male_perc,aged_18_to_19_years_male_perc,aged_20_to_24_years_male_perc,aged_25_to_29_years_male_perc,aged_30_to_34_years_male_perc,aged_35_to_39_years_male_perc,aged_40_to_44_years_male_perc,aged_45_to_49_years_male_perc,aged_50_to_54_years_male_perc,aged_55_to_59_years_male_perc,aged_60_to_64_years_male_perc,aged_65_years_male_perc,aged_66_to_69_years_male_perc,aged_70_to_74_years_male_perc,aged_75_to_79_years_male_perc,aged_80_to_84_years_male_perc,aged_85_years_and_over_male_perc,total_count,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count
0,E01011954,1200,42,41,40,33,82,11,31,22,70,76,96,83,61,80,91,81,70,8,48,49,25,32,28,3.500000,3.416667,3.333333,2.750000,6.833333,0.916667,2.583333,1.833333,5.833333,6.333333,8.000000,6.916667,5.083333,6.666667,7.583333,6.750000,5.833333,0.666667,4.000000,4.083333,2.083333,2.666667,2.333333,1080,40,43,51,42,81,18,26,19,73,77,75,56,69,47,78,80,71,9,31,42,20,18,14,3.703704,3.981481,4.722222,3.888889,7.500000,1.666667,2.407407,1.759259,6.759259,7.129630,6.944444,5.185185,6.388889,4.351852,7.222222,7.4074

In [11]:
# Create percentage columns for female_pop_df
total_col = 'total_count'
for col in combined_df.columns[-23:]:
    perc_col = col.replace('_count', '_perc')
    combined_df[perc_col] = (combined_df[col] / combined_df[total_col]) * 100
    
combined_df.head()

,lsoa21cd,total_female_count,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,total_male_count,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,aged_3_to_4_years_male_perc,aged_5_to_7_years_male_perc,aged_8_to_9_years_male_perc,aged_10_to_14_years_male_perc,aged_15_years_male_perc,aged_16_to_17_years_male_perc,aged_18_to_19_years_male_perc,aged_20_to_24_years_male_perc,aged_25_to_29_years_male_perc,aged_30_to_34_years_male_perc,aged_35_to_39_years_male_perc,aged_40_to_44_years_male_perc,aged_45_to_49_years_male_perc,aged_50_to_54_years_male_perc,aged_55_to_59_years_male_perc,aged_60_to_64_years_male_perc,aged_65_years_male_perc,aged_66_to_69_years_male_perc,aged_70_to_74_years_male_perc,aged_75_to_79_years_male_perc,aged_80_to_84_years_male_perc,aged_85_years_and_over_male_perc,total_count,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,aged_45_to_49_years_perc,aged_50_to_54_years_perc,aged_55_to_59_years_perc,aged_60_to_64_years_perc,aged_65_years_perc,aged_66_to_69_years_perc,aged_70_to_74_years_perc,aged_75_to_79_years_p

In [12]:
# Add total_female_perc and total_male_perc
combined_df['total_female_perc'] = (combined_df['total_female_count']/combined_df['total_count'])*100
combined_df['total_male_perc'] = (combined_df['total_male_count']/combined_df['total_count'])*100
combined_df.head()

,lsoa21cd,total_female_count,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,total_male_count,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,aged_3_to_4_years_male_perc,aged_5_to_7_years_male_perc,aged_8_to_9_years_male_perc,aged_10_to_14_years_male_perc,aged_15_years_male_perc,aged_16_to_17_years_male_perc,aged_18_to_19_years_male_perc,aged_20_to_24_years_male_perc,aged_25_to_29_years_male_perc,aged_30_to_34_years_male_perc,aged_35_to_39_years_male_perc,aged_40_to_44_years_male_perc,aged_45_to_49_years_male_perc,aged_50_to_54_years_male_perc,aged_55_to_59_years_male_perc,aged_60_to_64_years_male_perc,aged_65_years_male_perc,aged_66_to_69_years_male_perc,aged_70_to_74_years_male_perc,aged_75_to_79_years_male_perc,aged_80_to_84_years_male_perc,aged_85_years_and_over_male_perc,total_count,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,aged_45_to_49_years_perc,aged_50_to_54_years_perc,aged_55_to_59_years_perc,aged_60_to_64_years_perc,aged_65_years_perc,aged_66_to_69_years_perc,aged_70_to_74_years_perc,aged_75_to_79_years_p

In [13]:
#load population density data
pop_density_df = pd.read_excel(population_density_excel, sheet_name = 'Mid-2011 to mid-2022 LSOA 2021', skiprows = 3)

#drop columns
pop_density_df = pop_density_df[['LSOA 2021 Code','Mid-2011: Population','Mid-2011: People per Sq Km','Mid-2016: Population','Mid-2016: People per Sq Km','Mid-2021: Population','Mid-2021: People per Sq Km']]

#rename columns
pop_density_df.rename(columns = {'LSOA 2021 Code':'lsoa21cd','Mid-2011: Population':'mid_2011_population_count','Mid-2011: People per Sq Km':'mid_2011_people_per_sqkm','Mid-2016: Population':'mid_2016_population_count','Mid-2016: People per Sq Km':'mid_2016_people_per_sqkm','Mid-2021: Population':'mid_2021_population_count','Mid-2021: People per Sq Km':'mid_2021_people_per_sqkm'}, inplace = True)

#calculate density in hectares
pop_density_df['mid_2011_people_per_ha'] = pop_density_df['mid_2011_people_per_sqkm']/100
pop_density_df['mid_2016_people_per_ha'] = pop_density_df['mid_2016_people_per_sqkm']/100
pop_density_df['mid_2021_people_per_ha'] = pop_density_df['mid_2021_people_per_sqkm']/100

#drop pop density per sqkm columns
pop_density_df.drop(['mid_2011_people_per_sqkm','mid_2016_people_per_sqkm','mid_2021_people_per_sqkm'],1,inplace = True)                                                         

#Calculate population change
pop_density_df['change_in_population_2021_10yr'] = pop_density_df['mid_2021_population_count'] - pop_density_df['mid_2011_population_count']
pop_density_df['change_in_population_2021_5yr'] = pop_density_df['mid_2021_population_count'] - pop_density_df['mid_2016_population_count']

#Calculate population density change
pop_density_df['change_in_population_density_2021_10yr_change'] = pop_density_df['mid_2021_people_per_ha'] - pop_density_df['mid_2011_people_per_ha']
pop_density_df['change_in_population_density_2021_5yr_change'] = pop_density_df['mid_2021_people_per_ha'] - pop_density_df['mid_2016_people_per_ha']                                                          

pop_density_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\2090661237.py:16: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  pop_density_df.drop(['mid_2011_people_per_sqkm','mid_2016_people_per_sqkm','mid_2021_people_per_sqkm'],1,inplace = True)


,lsoa21cd,mid_2011_population_count,mid_2016_population_count,mid_2021_population_count,mid_2011_people_per_ha,mid_2016_people_per_ha,mid_2021_people_per_ha,change_in_population_2021_10yr,change_in_population_2021_5yr,change_in_population_density_2021_10yr_change,change_in_population_density_2021_5yr_change
0,E01000001,1472,1613,1573,113.405239,124.268105,121.186441,101,-40,7.781202,-3.081664
1,E01000002,1438,1289,1407,62.987297,56.460797,61.629435,-31,118,-1.357862,5.168638
2,E01000003,1348,1723,1610,228.862479,292.529711,273.344652,262,-113,44.482173,-19.185059
3,E01000005,987,1148,1104,52.056962,60.548523,58.227848,117,-44,6.170886,-2.320675
4,E01000006,1731,1882,1829,118.076398,128.376535,124.761255,98,-53,6.684857,-3.615280


In [14]:
# Combine the two dataframes on lsoa21cd and lsoa21nm
combined_df = combined_df.merge(pop_density_df, on=['lsoa21cd'])
combined_df.head()

,lsoa21cd,total_female_count,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,total_male_count,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,aged_3_to_4_years_male_perc,aged_5_to_7_years_male_perc,aged_8_to_9_years_male_perc,aged_10_to_14_years_male_perc,aged_15_years_male_perc,aged_16_to_17_years_male_perc,aged_18_to_19_years_male_perc,aged_20_to_24_years_male_perc,aged_25_to_29_years_male_perc,aged_30_to_34_years_male_perc,aged_35_to_39_years_male_perc,aged_40_to_44_years_male_perc,aged_45_to_49_years_male_perc,aged_50_to_54_years_male_perc,aged_55_to_59_years_male_perc,aged_60_to_64_years_male_perc,aged_65_years_male_perc,aged_66_to_69_years_male_perc,aged_70_to_74_years_male_perc,aged_75_to_79_years_male_perc,aged_80_to_84_years_male_perc,aged_85_years_and_over_male_perc,total_count,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,aged_45_to_49_years_perc,aged_50_to_54_years_perc,aged_55_to_59_years_perc,aged_60_to_64_years_perc,aged_65_years_perc,aged_66_to_69_years_perc,aged_70_to_74_years_perc,aged_75_to_79_years_p

In [15]:
#load number of households
num_households_df = pd.read_csv(num_households_csv)
num_households_df.drop(['Lower layer Super Output Areas'],1,inplace = True)
num_households_df.rename(columns = {'Lower layer Super Output Areas Code':'lsoa21cd','Observation':'2021_households_count'}, inplace = True)
num_households_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\2860853470.py:3: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  num_households_df.drop(['Lower layer Super Output Areas'],1,inplace = True)


,lsoa21cd,2021_households_count
0,E01000001,838
1,E01000002,824
2,E01000003,1017
3,E01000005,480
4,E01000006,554


In [16]:
# Combine the two dataframes on lsoa21cd and lsoa21nm
combined_df = combined_df.merge(num_households_df, on=['lsoa21cd'])
combined_df.head()

,lsoa21cd,total_female_count,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,total_male_count,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,aged_3_to_4_years_male_perc,aged_5_to_7_years_male_perc,aged_8_to_9_years_male_perc,aged_10_to_14_years_male_perc,aged_15_years_male_perc,aged_16_to_17_years_male_perc,aged_18_to_19_years_male_perc,aged_20_to_24_years_male_perc,aged_25_to_29_years_male_perc,aged_30_to_34_years_male_perc,aged_35_to_39_years_male_perc,aged_40_to_44_years_male_perc,aged_45_to_49_years_male_perc,aged_50_to_54_years_male_perc,aged_55_to_59_years_male_perc,aged_60_to_64_years_male_perc,aged_65_years_male_perc,aged_66_to_69_years_male_perc,aged_70_to_74_years_male_perc,aged_75_to_79_years_male_perc,aged_80_to_84_years_male_perc,aged_85_years_and_over_male_perc,total_count,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,aged_45_to_49_years_perc,aged_50_to_54_years_perc,aged_55_to_59_years_perc,aged_60_to_64_years_perc,aged_65_years_perc,aged_66_to_69_years_perc,aged_70_to_74_years_perc,aged_75_to_79_years_p

## 4. Load GIS shapefile 

In [17]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,FID,LSOA21CD,LSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E01000001,City of London 001A,129865.314476,2635.767993,4dd060c0-a44a-4c62-aca3-b0c88c0bf0d0,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,228419.782242,2707.816821,d0a47b5d-5649-4242-a8e7-462078975c1d,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,59054.204697,1224.573160,2904cc10-e994-4a6e-b11c-06be7fbd74d2,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,189577.709503,2275.805344,bfcf5958-f052-4388-8efd-75bb808d8b83,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,146536.995750,1966.092607,64a24862-79c1-4c8c-ab7b-d9cec1d3c0c6,"POLYGON ((545126.852 184310.838, 545145.213 18..."


## 5. GIS data management

### Remove Rename fields

In [18]:
#Drop and rename fields
lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\2026090638.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [19]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [20]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [21]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [22]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'inner', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [23]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_28480\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [24]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London


### Add data management fields

In [25]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [26]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700


# 7. Combine Geometry and data

In [27]:
census2021_population_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,combined_df, how = 'left', on='lsoa21cd')
census2021_population_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_female_count,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,total_male_count,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,aged_3_to_4_years_male_perc,aged_5_to_7_years_male_perc,aged_8_to_9_years_male_perc,aged_10_to_14_years_male_perc,aged_15_years_male_perc,aged_16_to_17_years_male_perc,aged_18_to_19_years_male_perc,aged_20_to_24_years_male_perc,aged_25_to_29_years_male_perc,aged_30_to_34_years_male_perc,aged_35_to_39_years_male_perc,aged_40_to_44_years_male_perc,aged_45_to_49_years_male_perc,aged_50_to_54_years_male_perc,aged_55_to_59_years_male_perc,aged_60_to_64_years_male_perc,aged_65_years_male_perc,aged_66_to_69_years_male_perc,aged_70_to_74_years_male_perc,aged_75_to_79_years_male_perc,aged_80_to_84_years_male_perc,aged_85_years_and_over_male_perc,total_count,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,aged_45_to_49_years_perc,aged_50_to_5

# 8. Final tweaks

In [28]:
#calculate household density
census2021_population_lsoa2021_gdb_df['2021_households_per_ha'] = census2021_population_lsoa2021_gdb_df['2021_households_count']/census2021_population_lsoa2021_gdb_df['area_ha']

#rename
census2021_population_lsoa2021_gdb_df.rename(columns = {'total_count':'2021_total_population_count','total_female_count':'2021_female_population_count', 'total_male_count':'2021_male_population_count','total_female_perc':'2021_female_population_perc','total_male_perc':'2021_male_population_perc' }, inplace = True)

census2021_population_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,2021_female_population_count,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,2021_male_population_count,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,aged_3_to_4_years_male_perc,aged_5_to_7_years_male_perc,aged_8_to_9_years_male_perc,aged_10_to_14_years_male_perc,aged_15_years_male_perc,aged_16_to_17_years_male_perc,aged_18_to_19_years_male_perc,aged_20_to_24_years_male_perc,aged_25_to_29_years_male_perc,aged_30_to_34_years_male_perc,aged_35_to_39_years_male_perc,aged_40_to_44_years_male_perc,aged_45_to_49_years_male_perc,aged_50_to_54_years_male_perc,aged_55_to_59_years_male_perc,aged_60_to_64_years_male_perc,aged_65_years_male_perc,aged_66_to_69_years_male_perc,aged_70_to_74_years_male_perc,aged_75_to_79_years_male_perc,aged_80_to_84_years_male_perc,aged_85_years_and_over_male_perc,2021_total_population_count,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,a

In [29]:
# Reorder columns in the combined dataframe

final_column_order = [
 'FID',
 'lsoa21cd',
 'lsoa21nm',
 'geometry',
 'msoa21cd',
 'msoa21nm',
 'wd22cd',
 'wd22nm',
 'lad22cd',
 'lad22nm', 
 'rgn22cd',
 'rgn22nm',
 'data_source',
 'data_resolution',
 'data_time_period',
 'data_web_link',
 'area_ha',    
 'mid_2011_population_count',
 'mid_2016_population_count',
 'mid_2021_population_count',
 '2021_total_population_count',  
 'mid_2011_people_per_ha',
 'mid_2016_people_per_ha',
 'mid_2021_people_per_ha',
 'change_in_population_2021_10yr',
 'change_in_population_2021_5yr',
 'change_in_population_density_2021_10yr_change',
 'change_in_population_density_2021_5yr_change',
 '2021_households_count',
 '2021_households_per_ha',
 '2021_female_population_count',
 '2021_male_population_count',
 '2021_female_population_perc',
 '2021_male_population_perc',
 'aged_2_years_and_under_count',
 'aged_3_to_4_years_count',
 'aged_5_to_7_years_count',
 'aged_8_to_9_years_count',
 'aged_10_to_14_years_count',
 'aged_15_years_count',
 'aged_16_to_17_years_count',
 'aged_18_to_19_years_count',
 'aged_20_to_24_years_count',
 'aged_25_to_29_years_count',
 'aged_30_to_34_years_count',
 'aged_35_to_39_years_count',
 'aged_40_to_44_years_count',
 'aged_45_to_49_years_count',
 'aged_50_to_54_years_count',
 'aged_55_to_59_years_count',
 'aged_60_to_64_years_count',
 'aged_65_years_count',
 'aged_66_to_69_years_count',
 'aged_70_to_74_years_count',
 'aged_75_to_79_years_count',
 'aged_80_to_84_years_count',
 'aged_85_years_and_over_count',
 'aged_2_years_and_under_perc',
 'aged_3_to_4_years_perc',
 'aged_5_to_7_years_perc',
 'aged_8_to_9_years_perc',
 'aged_10_to_14_years_perc',
 'aged_15_years_perc',
 'aged_16_to_17_years_perc',
 'aged_18_to_19_years_perc',
 'aged_20_to_24_years_perc',
 'aged_25_to_29_years_perc',
 'aged_30_to_34_years_perc',
 'aged_35_to_39_years_perc',
 'aged_40_to_44_years_perc',
 'aged_45_to_49_years_perc',
 'aged_50_to_54_years_perc',
 'aged_55_to_59_years_perc',
 'aged_60_to_64_years_perc',
 'aged_65_years_perc',
 'aged_66_to_69_years_perc',
 'aged_70_to_74_years_perc',
 'aged_75_to_79_years_perc',
 'aged_80_to_84_years_perc',
 'aged_85_years_and_over_perc',
  'aged_2_years_and_under_female_count',
 'aged_3_to_4_years_female_count',
 'aged_5_to_7_years_female_count',
 'aged_8_to_9_years_female_count',
 'aged_10_to_14_years_female_count',
 'aged_15_years_female_count',
 'aged_16_to_17_years_female_count',
 'aged_18_to_19_years_female_count',
 'aged_20_to_24_years_female_count',
 'aged_25_to_29_years_female_count',
 'aged_30_to_34_years_female_count',
 'aged_35_to_39_years_female_count',
 'aged_40_to_44_years_female_count',
 'aged_45_to_49_years_female_count',
 'aged_50_to_54_years_female_count',
 'aged_55_to_59_years_female_count',
 'aged_60_to_64_years_female_count',
 'aged_65_years_female_count',
 'aged_66_to_69_years_female_count',
 'aged_70_to_74_years_female_count',
 'aged_75_to_79_years_female_count',
 'aged_80_to_84_years_female_count',
 'aged_85_years_and_over_female_count',
 'aged_2_years_and_under_female_perc',
 'aged_3_to_4_years_female_perc',
 'aged_5_to_7_years_female_perc',
 'aged_8_to_9_years_female_perc',
 'aged_10_to_14_years_female_perc',
 'aged_15_years_female_perc',
 'aged_16_to_17_years_female_perc',
 'aged_18_to_19_years_female_perc',
 'aged_20_to_24_years_female_perc',
 'aged_25_to_29_years_female_perc',
 'aged_30_to_34_years_female_perc',
 'aged_35_to_39_years_female_perc',
 'aged_40_to_44_years_female_perc',
 'aged_45_to_49_years_female_perc',
 'aged_50_to_54_years_female_perc',
 'aged_55_to_59_years_female_perc',
 'aged_60_to_64_years_female_perc',
 'aged_65_years_female_perc',
 'aged_66_to_69_years_female_perc',
 'aged_70_to_74_years_female_perc',
 'aged_75_to_79_years_female_perc',
 'aged_80_to_84_years_female_perc',
 'aged_85_years_and_over_female_perc', 
 'aged_2_years_and_under_male_count',
 'aged_3_to_4_years_male_count',
 'aged_5_to_7_years_male_count',
 'aged_8_to_9_years_male_count',
 'aged_10_to_14_years_male_count',
 'aged_15_years_male_count',
 'aged_16_to_17_years_male_count',
 'aged_18_to_19_years_male_count',
 'aged_20_to_24_years_male_count',
 'aged_25_to_29_years_male_count',
 'aged_30_to_34_years_male_count',
 'aged_35_to_39_years_male_count',
 'aged_40_to_44_years_male_count',
 'aged_45_to_49_years_male_count',
 'aged_50_to_54_years_male_count',
 'aged_55_to_59_years_male_count',
 'aged_60_to_64_years_male_count',
 'aged_65_years_male_count',
 'aged_66_to_69_years_male_count',
 'aged_70_to_74_years_male_count',
 'aged_75_to_79_years_male_count',
 'aged_80_to_84_years_male_count',
 'aged_85_years_and_over_male_count',
 'aged_2_years_and_under_male_perc',
 'aged_3_to_4_years_male_perc',
 'aged_5_to_7_years_male_perc',
 'aged_8_to_9_years_male_perc',
 'aged_10_to_14_years_male_perc',
 'aged_15_years_male_perc',
 'aged_16_to_17_years_male_perc',
 'aged_18_to_19_years_male_perc',
 'aged_20_to_24_years_male_perc',
 'aged_25_to_29_years_male_perc',
 'aged_30_to_34_years_male_perc',
 'aged_35_to_39_years_male_perc',
 'aged_40_to_44_years_male_perc',
 'aged_45_to_49_years_male_perc',
 'aged_50_to_54_years_male_perc',
 'aged_55_to_59_years_male_perc',
 'aged_60_to_64_years_male_perc',
 'aged_65_years_male_perc',
 'aged_66_to_69_years_male_perc',
 'aged_70_to_74_years_male_perc',
 'aged_75_to_79_years_male_perc',
 'aged_80_to_84_years_male_perc',
 'aged_85_years_and_over_male_perc'
]

census2021_population_lsoa2021_gdb_df = census2021_population_lsoa2021_gdb_df[final_column_order]

census2021_population_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,mid_2011_population_count,mid_2016_population_count,mid_2021_population_count,2021_total_population_count,mid_2011_people_per_ha,mid_2016_people_per_ha,mid_2021_people_per_ha,change_in_population_2021_10yr,change_in_population_2021_5yr,change_in_population_density_2021_10yr_change,change_in_population_density_2021_5yr_change,2021_households_count,2021_households_per_ha,2021_female_population_count,2021_male_population_count,2021_female_population_perc,2021_male_population_perc,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,aged_45_to_49_years_perc,aged_50_to_54_years_perc,aged_55_to_59_years_perc,aged_60_to_64_years_perc,aged_65_years_perc,aged_66_to_69_years_perc,aged_70_to_74_years_perc,aged_75_to_79_years_perc,aged_80_to_84_years_perc,aged_85_years_and_over_perc,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,a

# 8. Create dominant field

In [30]:
# dominant sex 
# Determine the dominant sex based on population percentage difference

def determine_dominant_sex(row):
    if row['2021_female_population_perc'] > row['2021_male_population_perc'] + threshold_value:
        return 'female'
    elif row['2021_male_population_perc'] > row['2021_female_population_perc'] + threshold_value:
        return 'male'
    else:
        return 'neutral'

census2021_population_lsoa2021_gdb_df['dominant_sex'] = census2021_population_lsoa2021_gdb_df.apply(determine_dominant_sex, axis=1)

# Display the first few rows of the updated dataframe
census2021_population_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,mid_2011_population_count,mid_2016_population_count,mid_2021_population_count,2021_total_population_count,mid_2011_people_per_ha,mid_2016_people_per_ha,mid_2021_people_per_ha,change_in_population_2021_10yr,change_in_population_2021_5yr,change_in_population_density_2021_10yr_change,change_in_population_density_2021_5yr_change,2021_households_count,2021_households_per_ha,2021_female_population_count,2021_male_population_count,2021_female_population_perc,2021_male_population_perc,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,aged_45_to_49_years_perc,aged_50_to_54_years_perc,aged_55_to_59_years_perc,aged_60_to_64_years_perc,aged_65_years_perc,aged_66_to_69_years_perc,aged_70_to_74_years_perc,aged_75_to_79_years_perc,aged_80_to_84_years_perc,aged_85_years_and_over_perc,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,a

In [35]:
def determine_dominant_age_group(row):
    age_columns = [
        'aged_2_years_and_under_perc', 'aged_3_to_4_years_perc', 'aged_5_to_7_years_perc', 'aged_8_to_9_years_perc',
        'aged_10_to_14_years_perc', 'aged_15_years_perc', 'aged_16_to_17_years_perc', 'aged_18_to_19_years_perc',
        'aged_20_to_24_years_perc', 'aged_25_to_29_years_perc', 'aged_30_to_34_years_perc', 'aged_35_to_39_years_perc',
        'aged_40_to_44_years_perc', 'aged_45_to_49_years_perc', 'aged_50_to_54_years_perc', 'aged_55_to_59_years_perc',
        'aged_60_to_64_years_perc', 'aged_65_years_perc', 'aged_66_to_69_years_perc', 'aged_70_to_74_years_perc',
        'aged_75_to_79_years_perc', 'aged_80_to_84_years_perc', 'aged_85_years_and_over_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_population_lsoa2021_gdb_df['dominant_age_group'] = census2021_population_lsoa2021_gdb_df.apply(determine_dominant_age_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_population_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,mid_2011_population_count,mid_2016_population_count,mid_2021_population_count,2021_total_population_count,mid_2011_people_per_ha,mid_2016_people_per_ha,mid_2021_people_per_ha,change_in_population_2021_10yr,change_in_population_2021_5yr,change_in_population_density_2021_10yr_change,change_in_population_density_2021_5yr_change,2021_households_count,2021_households_per_ha,2021_female_population_count,2021_male_population_count,2021_female_population_perc,2021_male_population_perc,aged_2_years_and_under_count,aged_3_to_4_years_count,aged_5_to_7_years_count,aged_8_to_9_years_count,aged_10_to_14_years_count,aged_15_years_count,aged_16_to_17_years_count,aged_18_to_19_years_count,aged_20_to_24_years_count,aged_25_to_29_years_count,aged_30_to_34_years_count,aged_35_to_39_years_count,aged_40_to_44_years_count,aged_45_to_49_years_count,aged_50_to_54_years_count,aged_55_to_59_years_count,aged_60_to_64_years_count,aged_65_years_count,aged_66_to_69_years_count,aged_70_to_74_years_count,aged_75_to_79_years_count,aged_80_to_84_years_count,aged_85_years_and_over_count,aged_2_years_and_under_perc,aged_3_to_4_years_perc,aged_5_to_7_years_perc,aged_8_to_9_years_perc,aged_10_to_14_years_perc,aged_15_years_perc,aged_16_to_17_years_perc,aged_18_to_19_years_perc,aged_20_to_24_years_perc,aged_25_to_29_years_perc,aged_30_to_34_years_perc,aged_35_to_39_years_perc,aged_40_to_44_years_perc,aged_45_to_49_years_perc,aged_50_to_54_years_perc,aged_55_to_59_years_perc,aged_60_to_64_years_perc,aged_65_years_perc,aged_66_to_69_years_perc,aged_70_to_74_years_perc,aged_75_to_79_years_perc,aged_80_to_84_years_perc,aged_85_years_and_over_perc,aged_2_years_and_under_female_count,aged_3_to_4_years_female_count,aged_5_to_7_years_female_count,aged_8_to_9_years_female_count,aged_10_to_14_years_female_count,aged_15_years_female_count,aged_16_to_17_years_female_count,aged_18_to_19_years_female_count,aged_20_to_24_years_female_count,aged_25_to_29_years_female_count,aged_30_to_34_years_female_count,aged_35_to_39_years_female_count,aged_40_to_44_years_female_count,aged_45_to_49_years_female_count,aged_50_to_54_years_female_count,aged_55_to_59_years_female_count,aged_60_to_64_years_female_count,aged_65_years_female_count,aged_66_to_69_years_female_count,aged_70_to_74_years_female_count,aged_75_to_79_years_female_count,aged_80_to_84_years_female_count,aged_85_years_and_over_female_count,aged_2_years_and_under_female_perc,aged_3_to_4_years_female_perc,aged_5_to_7_years_female_perc,aged_8_to_9_years_female_perc,aged_10_to_14_years_female_perc,aged_15_years_female_perc,aged_16_to_17_years_female_perc,aged_18_to_19_years_female_perc,aged_20_to_24_years_female_perc,aged_25_to_29_years_female_perc,aged_30_to_34_years_female_perc,aged_35_to_39_years_female_perc,aged_40_to_44_years_female_perc,aged_45_to_49_years_female_perc,aged_50_to_54_years_female_perc,aged_55_to_59_years_female_perc,aged_60_to_64_years_female_perc,aged_65_years_female_perc,aged_66_to_69_years_female_perc,aged_70_to_74_years_female_perc,aged_75_to_79_years_female_perc,aged_80_to_84_years_female_perc,aged_85_years_and_over_female_perc,aged_2_years_and_under_male_count,aged_3_to_4_years_male_count,aged_5_to_7_years_male_count,aged_8_to_9_years_male_count,aged_10_to_14_years_male_count,aged_15_years_male_count,aged_16_to_17_years_male_count,aged_18_to_19_years_male_count,aged_20_to_24_years_male_count,aged_25_to_29_years_male_count,aged_30_to_34_years_male_count,aged_35_to_39_years_male_count,aged_40_to_44_years_male_count,aged_45_to_49_years_male_count,aged_50_to_54_years_male_count,aged_55_to_59_years_male_count,aged_60_to_64_years_male_count,aged_65_years_male_count,aged_66_to_69_years_male_count,aged_70_to_74_years_male_count,aged_75_to_79_years_male_count,aged_80_to_84_years_male_count,aged_85_years_and_over_male_count,aged_2_years_and_under_male_perc,a

# Special Case: Find and remove Duplicates

In [ ]:
def remove_duplicates(df, unique_column):
    """
    Removes duplicate rows based on a specified unique column while keeping the first occurrence.
    
    Parameters:
        df (GeoDataFrame or DataFrame): The input data to clean.
        unique_column (str): The column to check for duplicates.
    
    Returns:
        GeoDataFrame or DataFrame: A cleaned dataframe with duplicates removed.
    """
    # Check for duplicates
    duplicate_count = df.duplicated(subset=[unique_column], keep='first').sum()
    
    if duplicate_count > 0:
        print(f"Found {duplicate_count} duplicate(s) in column '{unique_column}'. Removing duplicates...")

        # Remove duplicates, keeping the first occurrence
        df = df.drop_duplicates(subset=[unique_column], keep='first')

        print(f"Duplicates removed. Now {unique_column} has only unique values.")

    else:
        print(f"No duplicates found in column '{unique_column}'. No changes made.")

    return df

# Specify the column where duplicates should be checked
unique_column = "lsoa21cd"

# Remove duplicates before uploading to PostGIS
census2021_age_by_single_year_msoa2021_gdb_df = remove_duplicates(census2021_age_by_single_year_msoa2021_gdb_df, unique_column)

census2021_age_by_single_year_msoa2021_gdb_df.head()

# 9. Upload to geodatabase

In [39]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_lsoa2021_pop_and_hholds"  # Desired table name
primary_key_column = "lsoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [40]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_population_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_population_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [41]:
# Publish the GeoDataFrame to PostGIS
census2021_population_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2021_lsoa2021_pop_and_hholds
Primary key set on column: lsoa21cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_lsoa2021_pop_and_hholds
